# Cloud Cover Threshold Comparison: RF vs SVM vs XGBoost

This notebook sweeps different CLOUDY_PIXEL_PERCENTAGE thresholds across all three
classifiers to demonstrate that **5% cloud cover** yields the best classification accuracy.

> Uses the same training points, bands, and classifier hyperparameters as the
> individual notebooks in `nb/rf/`, `nb/svm/`, and `nb/xgb/`.

In [ ]:
import os
from dotenv import load_dotenv
import ee
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Initialize Earth Engine
load_dotenv()
ee_project = os.getenv('EE_PROJECT_ID')
if not ee_project:
    raise ValueError("EE_PROJECT_ID not set in .env file")
ee.Initialize(project=ee_project)

SEED = 42
print("Earth Engine initialized successfully.")

### Campus Boundary

In [ ]:
campus_geojson = {
    "type": "Polygon",
    "coordinates": [
        [
            [80.01710357666015, 23.173962177472703],
            [80.03259601593017, 23.165361215115187],
            [80.03654422760009, 23.172502420044232],
            [80.026802444458, 23.181694681000845],
            [80.01542987823485, 23.176960548201308],
        ]
    ]
}
campus = ee.Geometry(campus_geojson)

### Cloud Mask & Helpers

In [ ]:
def mask_s2_clouds(image):
    qa = image.select('QA60')
    cloud = 1 << 10
    cirrus = 1 << 11
    mask = qa.bitwiseAnd(cloud).eq(0).And(qa.bitwiseAnd(cirrus).eq(0))
    return image.updateMask(mask).divide(10000)

bands = ['B4', 'B8', 'NDVI']

### Load Training Points

In [ ]:
forest_points = ee.FeatureCollection('users/ashutoshsaxena703/forest_points_train')
non_forest_points = ee.FeatureCollection('users/ashutoshsaxena703/non_forest_points_train')
training_points = forest_points.merge(non_forest_points)

print(f"Forest points:     {forest_points.size().getInfo()}")
print(f"Non-forest points: {non_forest_points.size().getInfo()}")

### Define Classifiers & Cloud Cover Thresholds

In [ ]:
CLASSIFIERS = {
    "Random Forest": lambda: ee.Classifier.smileRandomForest(
        numberOfTrees=10, minLeafPopulation=1, bagFraction=0.7
    ),
    "SVM (RBF)": lambda: ee.Classifier.libsvm(
        kernelType='RBF', gamma=0.5, cost=50
    ),
    "Gradient Boosted Trees": lambda: ee.Classifier.smileGradientTreeBoost(
        numberOfTrees=10, shrinkage=0.1, maxNodes=50
    ),
}

CLOUD_COVER_VALUES = [1, 2, 3, 5, 10, 15, 20, 30, 50]

print(f"Classifiers: {list(CLASSIFIERS.keys())}")
print(f"Cloud Cover Thresholds: {CLOUD_COVER_VALUES}")

### Run Cloud Cover Sweep

In [ ]:
results = []

for cc in CLOUD_COVER_VALUES:
    print(f"{'='*60}")
    print(f"  Cloud Cover Threshold: {cc}%")
    print(f"{'='*60}")

    # Build the image for this cloud cover threshold
    dataset = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
               .filterDate('2025-09-01', '2025-10-31')
               .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cc))
               .map(mask_s2_clouds))

    image_count = dataset.size().getInfo()
    print(f"  Available images: {image_count}")

    if image_count == 0:
        print(f"  ⚠ No images found for {cc}% threshold – skipping")
        for clf_name in CLASSIFIERS:
            results.append({
                'cloud_cover': cc, 'classifier': clf_name,
                'accuracy': None, 'kappa': None, 'image_count': 0,
            })
        continue

    image = dataset.median().clip(campus)

    # Add NDVI
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    image = image.addBands(ndvi)

    # Sample training data
    training = image.select(bands).sampleRegions(
        collection=training_points,
        properties=['label'],
        scale=10
    )
    training = training.filter(ee.Filter.notNull(bands + ['label']))

    # Train/test split (70/30)
    training = training.randomColumn('random', SEED)
    train_set = training.filter(ee.Filter.lt('random', 0.7))
    test_set  = training.filter(ee.Filter.gte('random', 0.7))

    for clf_name, clf_factory in CLASSIFIERS.items():
        try:
            classifier = clf_factory().train(
                features=train_set,
                classProperty='label',
                inputProperties=bands
            )

            validated = test_set.classify(classifier)
            cm = validated.errorMatrix('label', 'classification')
            accuracy = cm.accuracy().getInfo()
            kappa = cm.kappa().getInfo()

            print(f"  {clf_name:30s}  Accuracy: {accuracy:.4f}   Kappa: {kappa:.4f}")

            results.append({
                'cloud_cover': cc, 'classifier': clf_name,
                'accuracy': accuracy, 'kappa': kappa, 'image_count': image_count,
            })
        except Exception as e:
            print(f"  {clf_name:30s}  ERROR: {e}")
            results.append({
                'cloud_cover': cc, 'classifier': clf_name,
                'accuracy': None, 'kappa': None, 'image_count': image_count,
            })

print("✅ Sweep complete!")

### Results Table

In [ ]:
df = pd.DataFrame(results)
print(df[['cloud_cover', 'classifier', 'accuracy', 'kappa', 'image_count']].to_string(index=False))

### Accuracy & Kappa vs Cloud Cover

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = {
    "Random Forest": "#2ecc71",
    "SVM (RBF)": "#e74c3c",
    "Gradient Boosted Trees": "#3498db",
}

# --- Accuracy plot ---
ax1 = axes[0]
for clf_name in CLASSIFIERS:
    subset = df[df['classifier'] == clf_name].dropna(subset=['accuracy'])
    ax1.plot(subset['cloud_cover'], subset['accuracy'],
             marker='o', linewidth=2, label=clf_name,
             color=colors[clf_name])

ax1.axvline(x=5, color='#f39c12', linestyle='--', linewidth=1.5,
            alpha=0.7, label='5% threshold')
ax1.set_xlabel('Cloud Cover Threshold (%)', fontsize=12)
ax1.set_ylabel('Overall Accuracy', fontsize=12)
ax1.set_title('Classification Accuracy vs Cloud Cover Threshold', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_xticks(CLOUD_COVER_VALUES)

# --- Kappa plot ---
ax2 = axes[1]
for clf_name in CLASSIFIERS:
    subset = df[df['classifier'] == clf_name].dropna(subset=['kappa'])
    ax2.plot(subset['cloud_cover'], subset['kappa'],
             marker='s', linewidth=2, label=clf_name,
             color=colors[clf_name])

ax2.axvline(x=5, color='#f39c12', linestyle='--', linewidth=1.5,
            alpha=0.7, label='5% threshold')
ax2.set_xlabel('Cloud Cover Threshold (%)', fontsize=12)
ax2.set_ylabel('Kappa Statistic', fontsize=12)
ax2.set_title('Kappa vs Cloud Cover Threshold', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_xticks(CLOUD_COVER_VALUES)

plt.tight_layout()
plt.savefig('fe/cloud_cover_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fe/cloud_cover_comparison.png")

### Best Threshold Summary

In [ ]:
# Save CSV
df.to_csv('fe/cloud_cover_results.csv', index=False)
print("Saved: fe/cloud_cover_results.csv")

print("" + "="*70)
print("  BEST CLOUD COVER THRESHOLD PER CLASSIFIER")
print("="*70)
for clf_name in CLASSIFIERS:
    subset = df[(df['classifier'] == clf_name) & df['accuracy'].notna()]
    if not subset.empty:
        best = subset.loc[subset['accuracy'].idxmax()]
        print(f"  {clf_name:30s}  Best CC: {int(best['cloud_cover']):>3d}%"
              f"  Accuracy: {best['accuracy']:.4f}   Kappa: {best['kappa']:.4f}")

print("✅ Done!")